# All-sky camera: a measured cloud record for Cerro Pachón, 2021-07 → 2026-10

> **Diagnostic** supporting [ROSP-11](https://rubinobs.atlassian.net/browse/ROSP-11) (time-loss accounting); the allskycam is the primary cloud input of [weather_downtime_analysis.ipynb](weather_downtime_analysis.ipynb). Not an official metric page. Moved from [rubin_notebooks/earlyoperations/system_performance/allsky](https://github.com/bstalder/rubin_notebooks/tree/main/earlyoperations/system_performance/allsky), where the reduction pipeline still lives. Products are read from `ALLSKY_OUT` (default `/sdf/data/rubin/user/stalder/allsky`) and `SYSTEM_PERFORMANCE_DATA_DIR` (default `/sdf/data/rubin/user/stalder/system_performance_data`); results are written to the latter only with `ROSP11_WRITE_RESULTS=1` (`WRITE_RESULTS` in the first code cell).


The Canon 5D Mark IV + 8–15 mm fisheye all-sky camera ("allskycam" below, to keep it distinct from LSSTCam) has taken one 30 s frame per minute every night since July 2021. This notebook analyses the reduction described in [allsky/README.md](https://github.com/bstalder/rubin_notebooks/blob/main/earlyoperations/system_performance/allsky/README.md) (`allsky_reduce.py` → `regate_frames.py` → `build_completeness.py` → `cloud_metric.py` → `build_nightly.py`), whose products are read from `ALLSKY_OUT` (default: `/sdf/data/rubin/user/stalder/allsky`).

**Questions**
1. Is the allskycam's cloud index a faithful measurement of "sky unusable for Simonyi"? (Validation against the Scheduler `WEATHER` flag, Feb–Jul 2026.)
2. How does it compare with the two cloud records the weather-downtime notebook had to rely on: the meteoblue forecast nowcast and the CTIO 1975–2022 cloud database? And with DREAM?
3. What is the five-year cloud climatology at the site, and what should the scheduler simulations assume?
4. Side products: atmospheric transparency, zenith sky brightness, satellite/aircraft streak rates.

**The cloud index.** For every frame, every Yale-BSC star with V < 5.5, altitude > 20°, more than 25° from the moon is predicted (fisheye astrometry, ≈0.5 px rms) and checked for a detection. The *expected* number of detections is the sum of P(detect | V, alt, local sky brightness) from a clear-sky completeness model built per camera unit and calendar month from the clearest 35 % of that unit-month's frames (completeness tracks the allskycam's focus and plate scale, which both changed, and three bodies and three lenses were swapped in and out six times; see [allsky/README.md](https://github.com/bstalder/rubin_notebooks/blob/main/earlyoperations/system_performance/allsky/README.md)). **`cloud_idx` = 1 − observed/expected**, clipped to [0, 1]. `frac_sky_cloudy` is the fraction of NSIDE-8 HEALPix cells with index > 0.5. The moon enters only through the model's sky-brightness axis and the 25° exclusion.

> **Revision 2026-10-04.** The downtime notebook ([weather_downtime_analysis.ipynb](weather_downtime_analysis.ipynb), §4.1) found that on 425 nights the v1 running astrometric solution sat at 1.4–2.0 px rms and left a flat 0.3–0.6 cloud-index floor on clear skies (e.g. 2025-07-17). Those nights were re-reduced with v3, and the five-year fraction of dark time with index > 0.3 dropped from 0.24 to 0.18.
>
> **Revision 2026-10-11 (v5, every night re-reduced).** Three further problems were found and fixed, and all 1,682 nights were re-reduced: (1) 1,052 nights were still on v1 and failed astrometry on half of their full-moon frames; (2) the completeness model was per calendar month, but five months contain a camera swap, and the clear-sky reference (the frames with the most stars detected) then comes mostly from the sharper unit, so the softer unit's clear nights read as partly cloudy (in February 2023 the outgoing unit's 13 nights read 0.60 cloudy against a nowcast of 0.00); (3) the star finder's shape cuts rejected the soft off-axis star images of the defocused configurations (V<5 clear-sky completeness 0.67–0.84 → 0.90–1.00). Frames are now keyed by the true night (the 2021-07 → 2022-05 archive directories are UTC dates). The five-year fraction at index > 0.3 is now **0.149** (0.127 at index > 0.5), astrometry fails on 10 % of dark frames instead of 18 %, and agreement with the independent nowcast improved (r 0.75 → 0.80). The v3 products and notebook outputs are kept in `snapshot_v3_20261010/`.
>
> **Update 2026-10-28.** The record now runs to 2026-10-27: the allskycam was down 2026-07-19 → 10-09, and the 18 nights since its restart on 2026-10-10 (body 8323 + lens 0552) are reduced with v6, which tracks the lens's zoom creep in the astrometry and keys the completeness model per service stint. The five-year fraction at index > 0.3 is **0.152** (0.129 at index > 0.5). The rise from 0.149 is October 2026, which was cloudy (8 of 18 nights mostly lost), not a change to earlier nights. Products before the catch-up are in `snapshot_pre_catchup_20261028/`.
>
> **Update 2026-10-30.** The record runs to 2026-10-29. A **deep index** (Hipparcos V < 7.5) now runs alongside the standard one: it fills about 240 of the 272 cells above 20° instead of about 105, and it measures thin cloud as an extinction in magnitudes (§6.1). The standard index and every number above are unchanged.

In [ ]:
import glob, json, os, pathlib, warnings
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
# allskycam products and system-performance caches, built by the rubin_notebooks scripts.
A = pathlib.Path(os.environ.get("ALLSKY_OUT", "/sdf/data/rubin/user/stalder/allsky"))
SP = pathlib.Path(
    os.environ.get(
        "SYSTEM_PERFORMANCE_DATA_DIR",
        "/sdf/data/rubin/user/stalder/system_performance_data",
    )
)
# Result files (JSON / CSV / parquet) are written to SP only when WRITE_RESULTS is True,
# so a reviewer without write access to the cache can run the notebook end to end.
WRITE_RESULTS = os.environ.get("ROSP11_WRITE_RESULTS", "0") == "1"
FLAG0, FLAG1 = 20260225, 20260713
P = dict(
    blue="#2a78d6",
    orange="#eb6834",
    aqua="#1baf7a",
    yellow="#eda100",
    magenta="#e87ba4",
    green="#008300",
    violet="#4a3aa7",
    red="#e34948",
)
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"
mpl.rcParams.update(
    {
        "figure.facecolor": SURF,
        "axes.facecolor": SURF,
        "axes.edgecolor": GRID,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 1,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.labelcolor": INK2,
        "xtick.color": INK2,
        "ytick.color": INK2,
        "text.color": INK,
        "axes.titlecolor": INK,
        "axes.titlelocation": "left",
        "axes.titleweight": "semibold",
        "font.size": 10,
        "lines.linewidth": 2,
        "figure.dpi": 110,
    }
)
MON = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
N = pd.read_parquet(A / "index/allsky_nightly.parquet")  # one row per night
CL = pd.read_parquet(A / "index/cloud_all_frames.parquet")  # one row per reduced frame
CL["t"] = pd.to_datetime(CL.mjd + 2400000.5, unit="D", origin="julian", utc=True)
print(f"{len(N)} nights, {len(CL):,} reduced frames, {N.index.min()} → {N.index.max()}")

## 1. Coverage and reduction quality

In [ ]:
qa = N.groupby("year").agg(
    nights=("n_dark", "size"),
    dark_frames=("n_dark", "sum"),
    geometry_ok=("geom_ok", "mean"),
    clear_h_per_night=("clear_h", "mean"),
    disk_R_px=("disk_R", "median"),
)
qa["lens"] = pd.cut(
    qa.disk_R_px,
    [0, 1100, 1180, 1270, 1400],
    labels=["8 mm", "9 mm (a)", "9 mm (b)", "10 mm"],
)
qa.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.8))
m = N.groupby(N.index // 100).agg(n=("n_dark", "size"), ok=("geom_ok", "mean"))
x = pd.to_datetime(m.index.astype(str), format="%Y%m")
ax.bar(x, m.n, width=25, color=P["blue"], zorder=2, label="nights with frames")
ax.set_ylabel("nights / month")
ax.set_ylim(0, 33)
ax.set_title(
    "Archive coverage: disk 2021-07 → 2024-02, LFA 2024-02 → 2026-07 (gaps: Jul 2022, Jan 2023, Feb 2025)"
)
plt.tight_layout()

The astrometric gate "fails" on frames with too few stars to solve; those frames keep the night's running solution, so the cloud index is still defined. Geometry success is 90 % of dark frames after the v5 re-reduction (89 % in v3), and the unsolved frames are overcast. A *solved* frame can still carry a mis-registered solution (1.4–2.0 px rms) that inflates the index; after v5 no night has a median rms above 1.4 px and only 311 of 868,000 solved frames do (see the downtime notebook, §4.1). 2024 is the weakest year (86 %).

## 2. Does the cloud index measure what the observers see?

The Scheduler `WEATHER` bit (set by hand by the observers since 2026-02-25) is the operational truth. We compare frame by frame, excluding planned `DOWNTIME`.

In [ ]:
st = pd.read_parquet(SP / "scheduler_observatory_status.parquet")
st = st[st.salIndex == 1].sort_index()
c = CL[(CL.day_obs >= FLAG0) & (CL.day_obs <= FLAG1)].copy()
c["status"] = st.status.reindex(c.t, method="ffill").values
c = c[c.status.notna()]
c["status"] = c.status.astype(int)
c["W"] = (c.status & 8) > 0
c["OPER"] = c.status == 2
c["D"] = (c.status & 16) > 0
b = c[~c.D & c.cloud_idx.notna() & (c.sun_alt < -12)]
rows = []
for thr in [0.1, 0.2, 0.3, 0.4, 0.5, 0.7]:
    pr = b.cloud_idx > thr
    rows.append(
        dict(
            threshold=thr,
            P_cloudy_given_WEATHER=pr[b.W].mean(),
            P_cloudy_given_OPERATIONAL=pr[b.OPER].mean(),
            P_cloudy_given_OPER_moon_up=pr[b.OPER & (b.moon_alt > 20)].mean(),
            P_cloudy_given_WEATHER_moon_up=pr[b.W & (b.moon_alt > 20)].mean(),
        )
    )
print(
    f"{len(b):,} dark frames with Scheduler status on {b.day_obs.nunique()} nights; WEATHER {b.W.sum():,} frames, OPERATIONAL {b.OPER.sum():,}"
)
roc = pd.DataFrame(rows).set_index("threshold").round(3)
roc

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
bins = np.linspace(0, 1, 26)
axes[0].hist(
    b.cloud_idx[b.OPER],
    bins=bins,
    density=True,
    color=P["blue"],
    zorder=2,
    label="Scheduler OPERATIONAL",
)
axes[0].hist(
    b.cloud_idx[b.W],
    bins=bins,
    density=True,
    histtype="step",
    color=P["red"],
    lw=2,
    zorder=3,
    label="Scheduler WEATHER",
)
axes[0].set_xlabel("allskycam cloud index (1 − observed/expected stars)")
axes[0].set_ylabel("density")
axes[0].legend(frameon=False, fontsize=8.5)
axes[0].set_title("Frame-level separation, Feb–Jul 2026")
pn = (
    b.groupby("day_obs").agg(
        W_h=("W", "sum"), cam_h=("cloud_idx", lambda s: (s > 0.3).sum())
    )
    / 60
)
axes[1].plot([0, 12], [0, 12], color=GRID, lw=1)
axes[1].scatter(pn.cam_h, pn.W_h, s=26, color=P["blue"], edgecolor=SURF, lw=1, zorder=3)
axes[1].set_xlabel("allskycam hours with cloud index > 0.3")
axes[1].set_ylabel("Scheduler WEATHER hours")
axes[1].set_title(f"Per night, {len(pn)} nights (r = {pn.corr().iloc[0,1]:.2f})")
plt.tight_layout()
print(
    f"totals over these nights: WEATHER {pn.W_h.sum():.0f} h, allskycam cloudy (idx>0.3) {pn.cam_h.sum():.0f} h"
)

### 2.1 The disagreements

Two kinds. (a) **WEATHER set but the allskycam sees stars** (~30 % of flagged frames at threshold 0.3): the EFD limits, the dome state and the time since the last cloudy frame tell us what these are. (b) **OPERATIONAL but the allskycam sees cloud** (~7 % of operational frames): where and when.

In [ ]:
wt = pd.read_parquet(SP / "env_weather_tower_1min.parquet")
wt.index = pd.to_datetime(wt.index, utc=True)
wt["dpd"] = wt.air_temp - wt.dew_point
sh = pd.read_parquet(SP / "env_mtdome_shutter_1min.parquet")
sh.index = pd.to_datetime(sh.index, utc=True)
tm = b.t.dt.floor("1min")
m = wt.reindex(tm)
b = b.assign(
    lim=((m.rh > 80) | (m.dpd < 2) | (m.wind_speed > 15) | (m.wind_gust > 18)).values,
    open=(sh.shutter0.reindex(tm) > 95).values,
)
b = b.sort_values("t")
b["last_cloudy"] = b.t.where(b.cloud_idx > 0.3).groupby(b.day_obs).ffill()
b["min_since_cloudy"] = (b.t - b.last_cloudy).dt.total_seconds() / 60
cw = b[b.W & (b.cloud_idx <= 0.3)]
print(
    f"(a) WEATHER but allskycam-clear: {len(cw):,} frames = {len(cw)/b.W.sum():.0%} of WEATHER frames"
)
print(
    f"    outside RH/dew-point/wind limits: {cw.lim.mean():.0%} | shutter open: {cw.open.mean():.0%} | thin cloud 0.1<idx<=0.3: {(cw.cloud_idx > 0.1).mean():.0%}"
)
print(
    f"    minutes since the last allskycam-cloudy frame that night: median {cw.min_since_cloudy.median():.0f}, IQR {cw.min_since_cloudy.quantile(.25):.0f}–{cw.min_since_cloudy.quantile(.75):.0f}; never cloudy that night: {cw.min_since_cloudy.isna().mean():.0%}"
)
oc = b[b.OPER & (b.cloud_idx > 0.3)]
print(
    f"(b) OPERATIONAL but allskycam-cloudy: {len(oc):,} frames = {len(oc)/b.OPER.sum():.0%} of OPERATIONAL frames; partial sky (frac_sky_cloudy<0.5): {(oc.frac_sky_cloudy < 0.5).mean():.0%}; by month:",
    (oc.day_obs // 100).value_counts().sort_index().to_dict(),
)

**Reading.** The flagged-but-clear frames are mostly the observers *waiting*: 10 % are humidity/dew-point/wind closures, the shutter is closed in 96 % of them, 22 % show thin cloud just below the threshold (index 0.1–0.3), and the median frame is 88 minutes after the allskycam last saw cloud. The flag records the decision to stay closed, the allskycam records the sky. For a cloud *climatology* the allskycam is the right instrument; for *downtime accounting* the flag is, and the allskycam explains most of the difference between the two totals (154 h flagged vs 211 h allskycam-cloudy on the same nights: the allskycam also counts cloud during FAULT and during the first minutes before the flag is set). The operational-but-cloudy frames are concentrated in May–June 2026 and half are partial cover: the survey observing through broken cloud.

## 3. Allskycam vs the meteoblue nowcast and vs DREAM

The weather-downtime notebook had no sky measurement and used the meteoblue forecast valid for each hour (threshold 70 % total cloud cover, tuned to the flag) as a cloud proxy. The allskycam overlaps it on ~1000 nights.

In [ ]:
o = N[N.nowcast_cloud70_frac.notna()]
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
axes[0].plot([0, 1], [0, 1], color=GRID, lw=1)
axes[0].scatter(
    o.nowcast_cloud70_frac,
    o.cloudy30,
    s=10,
    alpha=0.5,
    color=P["blue"],
    edgecolor="none",
    zorder=3,
)
axes[0].set_xlabel("meteoblue nowcast: fraction of night with cloud cover > 70 %")
axes[0].set_ylabel("allskycam: fraction of night with cloud index > 0.3")
axes[0].set_title(
    f"Nightly, {len(o)} nights 2022-12 → 2026-07 (r = {o[['cloudy30','nowcast_cloud70_frac']].corr().iloc[0,1]:.2f})"
)
miss = ((o.nowcast_cloud70_frac < 0.1) & (o.cloudy30 > 0.5)).sum()
false = ((o.nowcast_cloud70_frac > 0.5) & (o.cloudy30 < 0.1)).sum()
axes[0].text(
    0.02,
    0.93,
    f"nowcast clear, allskycam cloudy: {miss} nights\nnowcast cloudy, allskycam clear: {false} nights",
    transform=axes[0].transAxes,
    fontsize=8.5,
    color=INK2,
    va="top",
)
d = N[N.dream_n.notna() & (N.dream_n > 10)]
axes[1].plot([0, 1], [0, 1], color=GRID, lw=1)
axes[1].scatter(
    d.dream_cloudy_frac,
    d.cloudy30,
    s=14,
    alpha=0.7,
    color=P["orange"],
    edgecolor="none",
    zorder=3,
)
axes[1].set_xlabel(
    "DREAM: fraction of night with >50 % of sky (alt>30°) at extinction > 0.1 mag"
)
axes[1].set_ylabel("allskycam: fraction with cloud index > 0.3")
axes[1].set_title(
    f"Allskycam vs DREAM, {len(d)} nights 2025-06 → 2026-06 (r = {d[['cloudy30','dream_cloudy_frac']].corr().iloc[0,1]:.2f})",
    fontsize=10,
)
plt.tight_layout()
print(
    f"same-night means: allskycam {o.cloudy30.mean():.3f}, nowcast {o.nowcast_cloud70_frac.mean():.3f}  → the nowcast under-reports cloud by a factor {o.cloudy30.mean()/o.nowcast_cloud70_frac.mean():.1f}"
)
print(
    f"DREAM nights: allskycam {d.cloudy30.mean():.3f} vs DREAM {d.dream_cloudy_frac.mean():.3f}"
)

**Reading.** The nowcast is one-sided: when it says cloudy the allskycam almost always agrees (8 exceptions in ~1000 nights), but it misses about one night in forty that the allskycam shows as mostly cloudy (25 nights), and it under-reports the cloudy fraction by a factor 1.2 (1.3 in v3, 1.9 before the v3 re-reduction; both earlier factors were partly reduction artefacts). This is exactly the failure mode you would expect from calibrating a forecast threshold to the operators' flag during one season. The downtime notebook's climatology, which rests on the nowcast outside 2026, is therefore biased low (§5). DREAM (305 nights) agrees with the allskycam on the clear/cloudy split at the night level (r ≈ 0.8); at DREAM's 0.1 mag threshold, which sits on its zero-point floor, it reports more cloudy time (0.19 vs 0.11). The downtime notebook (§4.5) compares the two frame by frame and cell by cell at a 0.5 mag threshold, where they agree.

## 4. Night classes from the downtime notebook, seen by the allskycam

In [ ]:
cc = (
    N[N["class"].notna()]
    .groupby("class")
    .agg(
        nights=("cloudy30", "size"),
        camera_cloudy_frac_median=("cloudy30", "median"),
        camera_cloudy_frac_mean=("cloudy30", "mean"),
    )
)
cc.round(2)

Weather-closed nights identified by the flag or the EFD proxy are 87–91 % cloudy (median) by the allskycam; hardware/engineering closures and observed nights have a median of 0 and a mean of 6–8 %. The "closed: weather (log text)" class is the exception: those are the humidity / dew-point closures of Dec 2025–Jan 2026 under a clear sky, correctly not seen as cloud. The nine "undetermined" nights were cloudy: the allskycam resolves them.

## 5. Five-year cloud climatology and the scheduler-simulation input

In [ ]:
clim = N.groupby("month").agg(
    cam20=("cloudy20", "mean"),
    cam30=("cloudy30", "mean"),
    cam50=("cloudy50", "mean"),
    idx_mean=("idx_mean", "mean"),
    nights=("cloudy30", "size"),
)
prev = pd.read_csv(SP / "weather_downtime_monthly_climatology.csv").set_index("month")
clim["nowcast_proxy_2023_26"] = prev.iloc[:, 1]
clim["CTIO_1975_2022"] = prev.iloc[:, 3]
clim["Scheduler_WEATHER_2026"] = prev.iloc[:, 0]
fig, ax = plt.subplots(figsize=(10, 4))
mo = np.arange(1, 13)
ax.fill_between(
    mo,
    clim.cam50 * 100,
    clim.cam20 * 100,
    color=P["blue"],
    alpha=0.15,
    lw=0,
    label="allskycam: index > 0.5 … > 0.2",
)
ax.plot(
    mo,
    clim.cam30 * 100,
    color=P["blue"],
    marker="o",
    ms=5,
    label="allskycam: fraction of dark time with cloud index > 0.3 (2021–26)",
)
ax.plot(
    mo,
    clim.CTIO_1975_2022 * 100,
    color=INK2,
    label="CTIO cloud DB 1975–2022, cloud > 3/8 (scheduler default)",
)
ax.plot(
    mo,
    clim.nowcast_proxy_2023_26 * 100,
    color=P["aqua"],
    label="meteoblue nowcast proxy 2023–26 (downtime notebook)",
)
ax.plot(
    mo,
    clim.Scheduler_WEATHER_2026 * 100,
    lw=0,
    marker="s",
    ms=7,
    color=P["orange"],
    label="Scheduler WEATHER flag, Mar–Jul 2026",
)
ax.set_xticks(mo)
ax.set_xticklabels(MON)
ax.set_ylabel("% of dark time")
ax.set_ylim(0, 60)
ax.legend(frameon=False, fontsize=8.5, loc="upper left")
ax.set_title("Seasonal cloud climatology at Cerro Pachón")
plt.tight_layout()
clim.round(3)

In [ ]:
yr = N.groupby("year").agg(
    nights=("cloudy30", "size"),
    cam30=("cloudy30", "mean"),
    fully=("cloudy30", lambda s: (s > 0.9).mean()),
    clear=("cloudy30", lambda s: (s < 0.1).mean()),
)
eqm = lambda col: N.groupby("month")[col].mean().mean()
head = pd.DataFrame(
    {
        "allskycam, index>0.3": dict(
            annual_mean_equal_month=eqm("cloudy30"),
            nightly_median=N.cloudy30.median(),
            nightly_std=N.cloudy30.std(),
            nights_fully_lost=(N.cloudy30 > 0.9).mean(),
            nights_partial=((N.cloudy30 >= 0.1) & (N.cloudy30 <= 0.9)).mean(),
            nights_clear=(N.cloudy30 < 0.1).mean(),
            annual_std_across_years=yr.cam30.std(),
        ),
        "allskycam, index>0.5": dict(
            annual_mean_equal_month=eqm("cloudy50"),
            nightly_median=N.cloudy50.median(),
            nightly_std=N.cloudy50.std(),
            nights_fully_lost=(N.cloudy50 > 0.9).mean(),
            nights_partial=((N.cloudy50 >= 0.1) & (N.cloudy50 <= 0.9)).mean(),
            nights_clear=(N.cloudy50 < 0.1).mean(),
            annual_std_across_years=N.groupby("year").cloudy50.mean().std(),
        ),
        "allskycam, mean index": dict(
            annual_mean_equal_month=eqm("idx_mean"),
            nightly_median=N.idx_mean.median(),
            nightly_std=N.idx_mean.std(),
        ),
    }
).T
print("by year (index>0.3):")
print(yr.round(3).to_string())
head.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
axes[0].hist(N.cloudy30, bins=np.linspace(0, 1, 21), color=P["blue"], zorder=2)
axes[0].set_xlabel("fraction of night with cloud index > 0.3")
axes[0].set_ylabel("nights")
axes[0].set_title("Nights are mostly clear or mostly lost")
yy = N.groupby(N.index // 100).cloudy30.mean()
x = pd.to_datetime(yy.index.astype(str), format="%Y%m")
axes[1].plot(x, yy * 100, color=P["blue"], lw=1.5)
axes[1].set_ylabel("% dark time cloudy (idx>0.3)")
axes[1].set_title(
    "Monthly series: 2023 was the cloudiest year, 2022 and 2026 the clearest"
)
plt.tight_layout()

### 5.1 Where the clouds are

Nightly mean cloud index per NSIDE-8 horizon pixel. Averaged over all nights the index is nearly flat with altitude (0.15 at the zenith, 0.17 at 10–60°) and with azimuth (0.14–0.18, with a weak southern maximum of 0.22). On nights the allskycam calls clear overall it is 0.02–0.06 in every direction except the south (0.10).

In v3 the same plot showed a strong north / north-west excess on clear nights (index 0.61 at N vs 0.04–0.17 elsewhere), and this section attributed it to the mast obstructing the horizon. That excess has gone. The per-pixel table behind it was built at 10:59 on 2026-10-04, before the v3 re-run finished (the completeness model was rebuilt at 14:38), and was not rebuilt afterwards (`build_index_tables.py` now does this in `finalize_rerun.sh`); the v1 mis-registered astrometry made the north/west half of the sky "cloudy" on 425 nights (downtime notebook §4.1), which is the likely source. It was a reduction artefact, not an obstruction. The residual southern excess is small, but it is present on clear nights too, so it should still be checked against the frames before any per-pixel use.

In [ ]:
import healpy as hp

PX = pd.read_parquet(A / "index/cloud_pix8_nightly.parquet")
mean_pix = PX.mean(axis=0).values
th, ph = hp.pix2ang(8, np.arange(768))
alt = 90 - np.degrees(th)
az = np.degrees(ph)
ok = np.isfinite(mean_pix) & (alt > 10)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
ab = pd.cut(alt[ok], [10, 20, 30, 45, 60, 90])
s = pd.Series(mean_pix[ok]).groupby(ab, observed=True).mean()
axes[0].bar(range(len(s)), s.values, color=P["blue"], zorder=2)
axes[0].set_xticks(range(len(s)))
axes[0].set_xticklabels([str(i) for i in s.index], fontsize=8)
axes[0].set_xlabel("altitude band (deg)")
axes[0].set_ylabel("mean cloud index, all nights")
axes[0].set_title("Cloud index vs altitude")
azb = pd.cut(
    az[ok & (alt < 45)],
    np.arange(0, 361, 45),
    labels=["N", "NE", "E", "SE", "S", "SW", "W", "NW"],
)
s2 = pd.Series(mean_pix[ok & (alt < 45)]).groupby(azb, observed=True).mean()
clear_n = N[N.cloudy30 < 0.05].index
mc = PX.loc[PX.index.isin(clear_n)].mean(axis=0).values
s3 = pd.Series(mc[ok & (alt < 45)]).groupby(azb, observed=True).mean()
w = 0.4
axes[1].bar(
    np.arange(len(s2)) - w / 2,
    s2.values,
    w,
    color=P["blue"],
    zorder=2,
    label="all nights",
)
axes[1].bar(
    np.arange(len(s2)) + w / 2,
    s3.values,
    w,
    color=P["orange"],
    zorder=2,
    label="allskycam-clear nights only",
)
axes[1].set_xticks(range(len(s2)))
axes[1].set_xticklabels(s2.index)
axes[1].set_xlabel("azimuth (alt < 45°)")
axes[1].set_title("Azimuth dependence, all nights vs clear nights")
axes[1].legend(frameon=False, fontsize=8.5)
plt.tight_layout()

## 6. Transparency, sky brightness and streaks

**Transparency.** On clear frames the matched stars give a zero point (V = m_inst + zp − k·X). Absolute zp depends on the camera unit and its plate scale (six body/lens swaps 2021–26, see `build_unit_history.py`), so transparency is expressed *relative to the clear-sky median zp of the same month and unit*; since the v5 re-reduction every night uses the same peak-based flux. Nights 0.1 mag below the monthly median are "non-photometric" by this allskycam (its own night-to-night scatter is 0.04 mag).

In [ ]:
FR = pd.read_parquet(
    A / "index/frames_all_regated.parquet",
    columns=[
        "day_obs",
        "seq",
        "zp",
        "zp_scatter",
        "n_phot",
        "pipe_version",
        "unit",
        "night",
        "sun_alt",
        "moon_alt",
        "fd5_far",
        "sky_zenith_adu_s",
    ],
)
FR = FR[(FR.pipe_version >= 5) & (FR.sun_alt < -18) & (FR.n_phot >= 100)].merge(
    CL[["day_obs", "seq", "cloud_idx"]], on=["day_obs", "seq"]
)  # one flux definition across the archive: v5 introduced the peak-based flux, v6 changed only the astrometry
clear = FR[FR.cloud_idx < 0.1]
nz = clear.groupby(["night", "unit"]).zp.median()
mon = nz.groupby(
    [nz.index.get_level_values("night") // 100, nz.index.get_level_values("unit")]
).transform(
    "median"
)  # absolute zp differs between camera units and focal lengths
dz = (nz - mon).rename("dzp").droplevel("unit")
dz = dz.dropna()
print(
    f"{len(dz)} nights with a clear-sky zero point; night-to-night scatter about the monthly median: {1.4826*np.median(np.abs(dz)):.3f} mag; nights > 0.1 mag below: {(dz < -0.1).mean():.1%}; > 0.2 mag: {(dz < -0.2).mean():.1%}"
)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
axes[0].hist(dz, bins=np.linspace(-0.6, 0.3, 46), color=P["blue"], zorder=2)
axes[0].set_xlabel("nightly clear-sky zp − monthly median (mag)")
axes[0].set_ylabel("nights")
axes[0].set_title("Transparency on allskycam-clear nights")
sk = N.sky_zenith_dark_med.dropna()
axes[1].scatter(
    pd.to_datetime(sk.index.astype(str), format="%Y%m%d"),
    sk,
    s=6,
    color=P["blue"],
    alpha=0.6,
    edgecolor="none",
)
axes[1].set_ylabel("zenith sky, moonless clear (ADU/s, green)")
axes[1].set_title(
    "Dark-sky brightness (uncalibrated; epoch steps are plate-scale changes)"
)
plt.tight_layout()

In [ ]:
# Streaks: interior (r/R<0.8, length>60 px) components on clear dark frames; rate per frame by year
import json

rows = []
fr_small = pd.read_parquet(
    A / "index/frames_all_regated.parquet",
    columns=["day_obs", "seq", "sun_alt", "disk_cx", "disk_cy", "disk_R"],
)
for f in sorted(glob.glob(str(A / "streaks/*.parquet")))[::2]:
    d = int(f.split("/")[-1][:8])
    s = pd.read_parquet(f)
    if len(s) == 0 or "x0" not in s:
        continue
    s["day_obs"] = d
    rows.append(s)
S = (
    pd.concat(rows)
    .merge(fr_small, on=["day_obs", "seq"], how="left")
    .merge(CL[["day_obs", "seq", "cloud_idx"]], on=["day_obs", "seq"], how="left")
)
S["r"] = (
    np.hypot((S.x0 + S.x1) / 2 - S.disk_cx, (S.y0 + S.y1) / 2 - S.disk_cy) / S.disk_R
)
inner = S[(S.r < 0.8) & (S.length_px > 60) & (S.cloud_idx < 0.1) & (S.sun_alt < -18)]
frames = CL[
    (CL.sun_alt < -18) & (CL.cloud_idx < 0.1) & CL.day_obs.isin(S.day_obs.unique())
]
rate = (
    inner.groupby(inner.day_obs // 10000).size()
    / frames.groupby(frames.day_obs // 10000).size()
).rename("interior streaks per clear dark frame")
tw = S[(S.r < 0.8) & (S.length_px > 60) & (S.cloud_idx < 0.1) & (S.sun_alt > -18)]
ftw = CL[
    (CL.sun_alt > -18)
    & (CL.sun_alt < -12)
    & (CL.cloud_idx < 0.1)
    & CL.day_obs.isin(S.day_obs.unique())
]
print(
    f"edge components (r/R>0.8: horizon structures, dome, mast): {(S.r > 0.8).mean():.0%} of all 'streaks' — excluded"
)
print(
    f"interior streak length: median {inner.length_px.median()*4.5/60:.0f}°, 90th pct {inner.length_px.quantile(.9)*4.5/60:.0f}° per 30 s frame (4.5′/px);  twilight rate {len(tw)/len(ftw):.2f} vs dark {len(inner)/len(frames):.2f} per frame"
)
pd.DataFrame(rate).round(2).T

Streak rates are a by-product and not yet validated against a satellite catalogue; the twilight excess (sunlit low-Earth-orbit satellites) and the sub-degree-per-second lengths are the right qualitative signatures. Treat the per-year trend as indicative only, since the detection threshold scales with the per-month noise.

### 6.1 Deep index: thin cloud and the empty half of the map

The standard index counts BSC stars with V < 5.5. On a clear dark frame that leaves only about 105 of the 272 NSIDE-8 cells above 20° with the 3 expected stars a cell needs, so the per-cell maps (§5.1) are undefined over 60 % of the sky. Those stars are also 1.5–2 mag brighter than the detection limit, so thin cloud barely changes their count: 0.5 mag of grey extinction removes about 1 % of them. The index sees thick cloud, not thin cloud.

After the 2026-10-16 refocus (star FWHM 1.4 px) the camera detects stars to V ≈ 7. A **deep index** now runs alongside the standard one. It uses the same detections and per-frame astrometry, with no CR2 re-read (`deep_match.py` → `build_completeness_deep.py` → `cloud_metric_deep.py`, see [allsky/README.md](https://github.com/bstalder/rubin_notebooks/blob/main/earlyoperations/system_performance/allsky/README.md)):

* catalog: Hipparcos V < 7.5 (19,700 stars), with proper motion applied and stars dropped that have a brighter neighbour within 0.3°. The match radius is 2 px.
* clear-sky model: per service stint × month as for the standard index, with finer V bins above V 6. Reference frames are ranked on the deep stars' own completeness, so thin-cloud frames are not used as "clear".
* `cloud_idx_deep` = 1 − observed/expected, and **`ext_deep`** = the grey extinction (mag, 0–3) that brings the model's expected count down to the observed one. Cell maps are at NSIDE 8 and NSIDE 16.

The standard index stays the five-year record. The deep stars are only well detected on the sharp stints, so the deep index's sensitivity changes with focus.

In [ ]:
DF = pd.read_parquet(A / "index/deep_cloud_all_frames.parquet")
TD = pd.read_parquet(A / "calib/clear_sky_completeness_deep.parquet")
dk = DF[DF.sun_alt < -18].copy()
dk["stint"] = dk.model_key.str.split("|").str[-1]
c0 = dk[dk.cloud_idx == 0]  # frames the standard index calls fully clear
v7 = TD[(TD.alt_lo >= 45) & (TD.sky_lo < 1.2) & (TD.Vlo == 7.0)].groupby("key")
deep_tab = pd.DataFrame(
    dict(
        dark_frames=dk.groupby("stint").size(),
        p_det_V7_clear=v7.n_det.sum() / v7.n.sum(),
        cells8_standard=c0.groupby("stint").n_cells.median(),
        cells8_deep=c0.groupby("stint").n_cells8.median(),
        cells16_deep=c0.groupby("stint").n_cells16.median(),
        clear_floor_idx=c0.groupby("stint").cloud_idx_deep.median(),
        clear_floor_ext=c0.groupby("stint").ext_deep.median(),
        thin_frac=dk.assign(t=(dk.cloud_idx < 0.1) & (dk.ext_deep > 0.2))
        .groupby("stint")
        .t.mean(),
    )
).dropna(subset=["dark_frames"])
# DREAM whole-sky extinction at each allskycam frame (nearest map within 60 s), 2025-06 -> 2026-06
dm = pd.read_parquet(
    SP / "dream_maps.parquet", columns=["mjd", "med_ext_alt30"]
).sort_values("mjd")
dd = pd.merge_asof(
    dk[["mjd", "cloud_idx", "cloud_idx_deep", "ext_deep"]]
    .dropna(subset=["mjd"])
    .sort_values("mjd"),
    dm,
    on="mjd",
    tolerance=60 / 86400,
    direction="nearest",
).dropna(subset=["med_ext_alt30", "ext_deep"])
thin = dd[dd.cloud_idx < 0.05]
EB = [0, 0.05, 0.1, 0.2, 0.4, 0.8, 3.0]
tb = thin.groupby(
    pd.cut(thin.ext_deep, EB, include_lowest=True), observed=True
).med_ext_alt30
tb = pd.DataFrame(
    dict(
        frames=tb.size(),
        dream_median=tb.median(),
        dream_q25=tb.quantile(0.25),
        dream_q75=tb.quantile(0.75),
    )
)
print(
    f"{len(dk):,} dark frames; on clear frames the deep index covers {c0.n_cells8.median():.0f} of 272 NSIDE-8 cells "
    f"(standard {c0.n_cells.median():.0f}); {((dk.cloud_idx < 0.1) & (dk.ext_deep > 0.2)).mean():.1%} of dark frames are "
    f"clear by the standard index but have ext_deep > 0.2 mag"
)
print(
    f"DREAM overlap: {len(dd):,} frames; frames with standard index < 0.05, DREAM median extinction by ext_deep bin:"
)
display(tb.round(3))
display(deep_tab.round(3))

fig, axes = plt.subplots(
    1, 2, figsize=(10.5, 3.9), gridspec_kw=dict(width_ratios=[1, 1.1])
)
cov = deep_tab[["cells8_standard", "cells8_deep"]].iloc[::-1] / 272
y = np.arange(len(cov))
h = 0.4
axes[0].barh(
    y + h / 2,
    cov.cells8_standard,
    h * 0.9,
    color=P["blue"],
    zorder=2,
    label="standard (BSC V<5.5)",
)
axes[0].barh(
    y - h / 2,
    cov.cells8_deep,
    h * 0.9,
    color=P["orange"],
    zorder=2,
    label="deep (Hipparcos V<7.5)",
)
axes[0].set_yticks(y)
axes[0].set_yticklabels(
    [
        f"{s.split('@')[0]}  from {pd.Timestamp(s.split('@')[1]):%Y-%m-%d}"
        for s in cov.index
    ],
    fontsize=8,
)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("fraction of alt>20° NSIDE-8 cells with ≥ 3 expected stars")
axes[0].set_title("Sky coverage on clear dark frames, by stint")
axes[0].legend(
    frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2
)
axes[0].grid(axis="y", visible=False)
xc = np.arange(len(tb))
axes[1].fill_between(
    xc,
    tb.dream_q25,
    tb.dream_q75,
    color=P["orange"],
    alpha=0.2,
    lw=0,
    zorder=2,
    label="DREAM interquartile range",
)
axes[1].plot(
    xc,
    tb.dream_median,
    color=P["orange"],
    marker="o",
    ms=8,
    zorder=3,
    label="DREAM median",
)
axes[1].set_xticks(xc)
axes[1].set_xticklabels([f"{a:g}–{b:g}" for a, b in zip(EB[:-1], EB[1:])], fontsize=8)
for xi, (n, v) in enumerate(zip(tb.frames, tb.dream_median)):
    axes[1].annotate(
        f"{n:,}",
        (xi, v),
        textcoords="offset points",
        xytext=(0, 9),
        ha="center",
        fontsize=7.5,
        color=INK2,
    )
axes[1].set_xlabel("allskycam ext_deep (mag); labels: frames per bin")
axes[1].set_ylabel("DREAM extinction, alt>30° (mag)")
axes[1].set_title("Frames the standard index calls clear: DREAM vs ext_deep")
axes[1].legend(frameon=False, fontsize=8.5, loc="upper left")
plt.tight_layout()

**Reading.** On clear dark frames the deep index covers about 240 of the 272 cells, against about 105 for the standard one, in every stint. At NSIDE 16 (3.7° cells) it covers 460–725 cells on the sharp stints (8323/0552 since 2026-10-16, 9118/0165 in 2022, 9127/0545 in 2023). Its clear-sky floor is a median index of 0.01–0.02 (≈ 0.02 mag) in every stint except the six-night 8323/0552@20261010 stint (0.09), which never had enough clear frames for its own reference.

Thin cloud: about 6 % of dark frames are clear by the standard index but have `ext_deep` > 0.2 mag. DREAM agrees on these frames. Where `ext_deep` is below 0.2 mag the DREAM median is 0.04 mag, at 0.2–0.4 mag it is 0.07 mag, and at 0.4–0.8 mag it is 0.28 mag. In October 2026, 11–38 % of the dark frames on 10-17, 10-22 and 10-28 were thin cloud of this kind, all nights the standard index calls clear.

Limits: completeness at V 7.0–7.25 on a clear dark sky is 0.76–0.84 on the two sharpest stints (8323/0552 since 2026-10-16, 9118/0165 in 2022), 0.51 for 9127/0545, and 0.02–0.23 on the others (FWHM 2.6–3.5 px). On those stints the deep index is close to a V < 6 index, and `ext_deep` saturates near 1 mag. Frame by frame it does not track DREAM better than the standard index for thick cloud (rank correlation with DREAM extinction 0.42 vs 0.43). Its gain is coverage and thin cloud. Per-frame values are in `index/deep_cloud_all_frames.parquet`, nightly values in `index/deep_nightly.parquet`.

## 7. Revising the weather-downtime inputs

The downtime notebook estimated the nightly fraction of dark time lost to weather with a nowcast-based proxy (equal-month mean 0.155) and compared it with the CTIO cloud database (0.254). The allskycam says the nowcast under-reports cloud by ~1.2×. Replacing it:

In [ ]:
win = N.loc[20250415:20260713]
limh = win.efd_lim_frac.fillna(0) * win.dark_h
cam_only = win.cloudy30
combined = np.clip(
    win.cloudy30 + win.efd_lim_frac.fillna(0) * (1 - win.cloudy30), 0, 1
)  # cloud OR RH/dew/wind limit (limits mostly coincide with cloud)
res = pd.DataFrame(
    {
        "nowcast proxy (downtime notebook)": dict(
            window_mean=win.nowcast_proxy_bad_frac.mean(),
            equal_month_mean_full_baseline=prev.iloc[:, 1].mean(),
        ),
        "allskycam cloud idx>0.3": dict(
            window_mean=cam_only.mean(), equal_month_mean_full_baseline=eqm("cloudy30")
        ),
        "allskycam cloud ∪ EFD limits": dict(
            window_mean=combined.mean(),
            equal_month_mean_full_baseline=N.assign(
                c=np.clip(
                    N.cloudy30 + N.efd_lim_frac.fillna(0) * (1 - N.cloudy30), 0, 1
                )
            )
            .groupby("month")
            .c.mean()
            .mean(),
        ),
        "CTIO cloud DB (sim default)": dict(
            window_mean=np.nan, equal_month_mean_full_baseline=prev.iloc[:, 3].mean()
        ),
    }
).T
print(
    f"LSSTCam window {win.index.min()}–{win.index.max()}: {len(win)} nights; Scheduler flag on the 2026 part gave {win.W_h.sum():.0f} h = {win.W_h.sum()/win.night_h.sum():.3f} of dark time"
)
res.round(3)

**Recommendation for the scheduler simulations (supersedes §5.2 of the downtime notebook's pre-allskycam version; the downtime notebook's own §5.2 now carries the same numbers).**

* The allskycam gives a *measured* fraction of dark time with unusable sky of **≈ 0.15** (equal-month annual mean 0.152 at index > 0.3; 0.17 at index > 0.2, 0.13 at index > 0.5; 0.17 with the humidity / dew-point / wind limits added). This is well below the CTIO 1975–2022 default (0.25) and the like-for-like comparison with the nowcast-based 0.155 (which includes the limits) is the allskycam ∪ limits value of 0.169: after v5 the two differ by only 0.01 annually and month by month by at most 0.06 (May), so the nowcast proxy is no longer badly biased on average. The allskycam is still the better input: it measures the sky directly, it covers 2021–22 where the nowcast does not exist, and it is validated against LSSTCam photometry (downtime notebook §4.1). Against the 2026 flag the two are equally good (night-level r 0.75 vs 0.76, downtime notebook §4.2), as expected since the nowcast threshold was tuned to the flag.
* Seasonality is strong and the allskycam's monthly curve (table above) is the right input: June ≈ 0.39, Aug ≈ 0.31, May ≈ 0.25, Jan–Mar ≈ 0.02–0.03. The scheduler default's seasonal shape differs: it puts 3–5× too much cloud in Jan–Mar and 2× too much in Sep–Nov, and is about right only in Jun and Aug. The secondary February peak the v3 reduction showed (0.11) was the camera-swap artefact of February 2023 and is gone.
* Nights are bimodal: 74 % clear (<10 % cloudy), 8 % fully lost (>90 %), 18 % partial. The year-to-year standard deviation of the annual fraction is ≈ 0.04 over six years (by year 0.11–0.20; 2021 the clearest, 2024 the cloudiest; 2026 is a partial year, Jan–Jul and Oct), smaller than the CTIO estimate of 0.05.
* Operators observed through part of this cloud: at the flag's own threshold the allskycam-cloudy hours exceed WEATHER hours by ~37 % in 2026 (211 vs 154 h). A simulation that models "closed when cloud index > 0.3" is pessimistic by that margin relative to actual 2026 practice; "index > 0.5" (0.13 annual) matches the flagged hours more closely. Both numbers are reported above so the choice is explicit.

**Caveats.** The cloud index is a star-count metric: thin uniform cirrus that dims stars without hiding them registers weakly; the transparency series (§6) and the LSSTCam zero-point comparison in the downtime notebook (§4.1) are the complementary measures. Under a bright moon (sky > 12 ADU/s, 6 % of dark frames) the completeness model still raises the index above 0.3 on 6 % of photometric LSSTCam exposures, against 0.1 % otherwise; the 2025-12-01 → 03 nights that showed this floor in v3 now have no frame above index 0.3. Frames with the sun above −12° are not reduced, and about 19,000 dark frames taken at 1 s or 1/5000 s (mostly 2024-04 → 09) cannot be used.

In [ ]:
out = dict(
    nights=int(len(N)),
    frames=int(len(CL)),
    flag_validation=roc.to_dict(),
    monthly_climatology=clim.round(4).to_dict(),
    headline=head.round(4).to_dict(orient="index"),
    by_year=yr.round(4).to_dict(orient="index"),
    downtime_revision=res.round(4).to_dict(orient="index"),
)
files = [
    "allsky_cloud_results.json",
    "allsky_monthly_cloud_climatology.csv",
    "allsky_nightly_summary.parquet",
]
if WRITE_RESULTS:
    json.dump(out, open(SP / files[0], "w"), indent=1, default=str)
    clim.to_csv(SP / files[1])
    N.to_parquet(SP / files[2])
    print(f"saved to {SP}: " + ", ".join(files))
else:
    print(f"not saved (set ROSP11_WRITE_RESULTS=1 to write {', '.join(files)} to {SP})")